Installing necessary libraries

In [1]:
%pip install geoclip
%pip install exif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 54.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 77.7 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geoclip]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [exif]


Import Dependencies

In [2]:
import torch
from exif import Image
from geoclip import GeoCLIP
import matplotlib.pyplot as plt
import numpy as np
import math
from pathlib import Path

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Functions Useful for coordinate math

In [27]:
def dms_to_decimal(coords, ref):
    degrees, minutes, seconds = coords
    decimal = degrees + (minutes / 60.0) + (seconds / 3600.0)
    if ref in ['S', 'W']:
        decimal = -decimal
    return decimal

def getCoordinateFromExif(image:Path):
  with open(image, 'rb') as image_file:
    img = Image(image_file)
    try:
      lat = dms_to_decimal(img.gps_latitude, img.gps_latitude_ref)
      lon = dms_to_decimal(img.gps_longitude, img.gps_longitude_ref)
      return coordinate(lat, lon)
    except AttributeError:
      return None


class coordinate():
  def __init__(self, x, y):
    self.lat = x
    self.lon = y



def haversine(coord1:coordinate, coord2:coordinate):
  R = 3956.0

  lat1 = math.radians(coord1.lat)
  lon1 = math.radians(coord1.lon)
  lat2 = math.radians(coord2.lat)
  lon2 = math.radians(coord2.lon)

  dlat = lat2 - lat1
  dlon = lon2 - lon1

  a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
  c = 2 * math.asin(math.sqrt(a))

  return R * c



class coordImage():
  def __init__(self, path):
    self.path = path
    self.coord = getCoordinateFromExif(path)
    self.accurate = None
    self.prob = None
    self.dist = None

  def getClosestPredOutOfK(self, model, k:int, accuracy_threshold:int):
    if self.coord == None:
      return None, None
    top_pred_gps, top_pred_prob = model.predict(str(self.path), top_k=k)
    minDist = math.inf
    minIdx = None
    for i in range(k):
      estimatedCoord = coordinate(*top_pred_gps[i])
      dist = haversine(self.coord, estimatedCoord)
      minDist = min(minDist, dist)
      minIdx = i if dist == minDist else minIdx
    self.dist = minDist
    self.prob = top_pred_prob[minIdx]
    self.accurate = minDist <= accuracy_threshold
    return minDist, top_pred_prob[minIdx]


In [28]:
def buildImageObjects(source:Path):
  total = 0
  accurate = 0
  model = GeoCLIP().to("cuda")
  images = []
  for img in source.rglob("*"):
    if not img.is_file():
      print(str(img) + " is not a file")
      continue
    imObject = coordImage(img)
    minDist, minProb = imObject.getClosestPredOutOfK(model, 5, 5)
    if minDist == None:
      continue
    images.append(imObject)
    total += 1
    accurate += 1 if imObject.accurate else 0
    if (total % 10) == 0:
      print("Processed:", total)
      print("Accuracy:", accurate / total)

  return accurate/total, images

In [31]:
def evaluate(source:Path):
  baselineAccuracy, images = buildImageObjects(source)
  highDistinctive = []
  mediumDistinctive = []
  lowDistinctive = []
  Accuracies = {
      "High": 0,
      "Medium": 0,
      "Low": 0
  }
  Totals = {
      "High": 0,
      "Medium": 0,
      "Low": 0
  }
  for im in images:
    imClass = None
    if "High" in str(im.path):
      highDistinctive.append(im)
      imClass = "High"
    elif "Medium" in str(im.path):
      mediumDistinctive.append(im)
      imClass = "Medium"
    else:
      lowDistinctive.append(im)
      imClass = "Low"

    if im.coord:
      Totals[imClass] += 1
      Accuracies[imClass] += 1 if im.accurate else 0

  return baselineAccuracy, Accuracies, Totals

In [33]:
image_source = Path("/content/drive/MyDrive/CS117 Project Images")
basline, accurcies, totals = evaluate(image_source)
print("Baseline Accuracy:", basline)
print("High Distinctive Accuracy:", accurcies["High"]/totals["High"])
print("Medium Distinctive Accuracy:", accurcies["Medium"]/totals["Medium"])
print("Low Distinctive Accuracy:", accurcies["Low"]/totals["Low"])

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

/content/drive/MyDrive/CS117 Project Images/Highly Distinctive is not a file
/content/drive/MyDrive/CS117 Project Images/Medium Distinctive is not a file
/content/drive/MyDrive/CS117 Project Images/Not Distinctive is not a file
Processed: 10
Accuracy: 0.8
Processed: 20
Accuracy: 0.9
Processed: 30
Accuracy: 0.9


/usr/local/lib/python3.13/dist-packages/exif/_app1_metadata.py:475: RuntimeWarning: skipping bad IFD gps
  warnings.warn(f"skipping bad IFD {ifd_key}", RuntimeWarning)


Processed: 40
Accuracy: 0.875
Processed: 50
Accuracy: 0.86
Processed: 60
Accuracy: 0.8833333333333333
Processed: 70
Accuracy: 0.8714285714285714
Processed: 80
Accuracy: 0.875
Processed: 90
Accuracy: 0.8777777777777778
Processed: 100
Accuracy: 0.87
Processed: 110
Accuracy: 0.8636363636363636
Processed: 120
Accuracy: 0.8166666666666667
Processed: 130
Accuracy: 0.8
Processed: 140
Accuracy: 0.7714285714285715
Processed: 150
Accuracy: 0.76
Processed: 160
Accuracy: 0.75625
Processed: 170
Accuracy: 0.7470588235294118
Processed: 180
Accuracy: 0.7333333333333333
Processed: 190
Accuracy: 0.7263157894736842
Processed: 200
Accuracy: 0.705
Processed: 210
Accuracy: 0.6904761904761905
Processed: 220
Accuracy: 0.6818181818181818
Processed: 230
Accuracy: 0.6782608695652174
Processed: 240
Accuracy: 0.6666666666666666
Processed: 250
Accuracy: 0.66
Processed: 260
Accuracy: 0.6384615384615384
Processed: 270
Accuracy: 0.6259259259259259
Processed: 280
Accuracy: 0.6178571428571429
Processed: 290
Accuracy: 0.